# TFP-BD My 80/20 split: all 5 detectors vs 70/20/10 (same test set)

Amar 80/20 split e train kora 5 ta detector (YOLOv8n, YOLO11n, RT-DETR-L, Faster R-CNN, EfficientDet-D3) ke
**70/20/10 er HUBOHU same test set e** chalano hoy, ar purono model er sathe **ekই frame e jora dhore** tulona:

- Detection: mAP50, mAP50-95, class-wise AP50 (Part 2 er moto COCO mAP)
- Encroachment: same 1,605 test frame; notun model er threshold 80/10/10 validation e bachai
- Congestion: same 2,368 test chobi, same level (Low <= 7, Medium <= 11)
- Purono vs notun: McNemar test + bootstrap 95% CI of the difference

**Input:** Bangladeshi Traffic Flow Dataset, Data Preparation (Sharmin), **TFPBD_Final_Evaluation** (Part 1 output),
**TFPBD_Final_Evaluation2** (ager 80/20 evaluation: YOLO + RT-DETR er result `dets_8010.pkl` ekhan theke ney, abar chalate hoy na),
**Faster R-CNN 80/20** ar **EfficientDet-D3 80/20** training notebook er output. GPU T4 x2 (na thakle CPU, slow), Internet ON.

YOLOv8n / YOLO11n / RT-DETR-L er result ager run theke hubohu nawa hoy; shudhu notun 2 ta model er inference chole.

In [ ]:
# CELL 1: install
!pip install -q ultralytics torchmetrics pycocotools effdet
import ultralytics, torchmetrics
print("ultralytics", ultralytics.__version__, "| torchmetrics", torchmetrics.__version__)

In [ ]:
# CELL 2: config + input khuje ber kora
import os, re, glob, time, math, pickle, random, gc, json, zipfile, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from matplotlib.path import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

WK = "/kaggle/working"
OUTD = f"{WK}/eval_8020_all_results"
os.makedirs(OUTD, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(GPU nai!)")

CLASSES = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
PED = {8}
LOC_NAME = {1: "Arambag", 2: "Shapla Chattar", 3: "Abul Hotel", 4: "Bashabo"}
EXTS = (".jpg", ".jpeg", ".png", ".bmp")
THRS = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]
ALIAS = {"bike": "Motorcycle", "motorcycle": "Motorcycle", "cycle": "Bicycle", "bicycle": "Bicycle",
         "people": "Pedestrian", "pedestrian": "Pedestrian", "cng": "CNG", "car": "Car", "bus": "Bus",
         "truck": "Truck", "rickshaw": "Rickshaw", "mini-truck": "Mini-Truck"}

# purono (70/20/10) model: Part 2 er final threshold ar mAP
ENC_OLD = {"YOLOv8n": 0.35, "YOLO11n": 0.30, "RT-DETR-L": 0.60, "Faster R-CNN": 0.55, "EfficientDet-D3": 0.35}
CONG_OLD = {"YOLOv8n": 0.35, "YOLO11n": 0.35, "RT-DETR-L": 0.60, "Faster R-CNN": 0.70, "EfficientDet-D3": 0.40}
MAP_OLD = {"YOLOv8n": {"mAP50": 0.7323, "mAP50-95": 0.5044, "AP50_Car": 0.9257, "AP50_Bus": 0.8763, "AP50_Truck": 0.4525, "AP50_Motorcycle": 0.8366, "AP50_Bicycle": 0.1963, "AP50_Rickshaw": 0.9012, "AP50_Mini-Truck": 0.6424, "AP50_CNG": 0.8972, "AP50_Pedestrian": 0.8621}, "YOLO11n": {"mAP50": 0.71, "mAP50-95": 0.4841, "AP50_Car": 0.9201, "AP50_Bus": 0.8698, "AP50_Truck": 0.4149, "AP50_Motorcycle": 0.8188, "AP50_Bicycle": 0.1135, "AP50_Rickshaw": 0.8932, "AP50_Mini-Truck": 0.6244, "AP50_CNG": 0.8903, "AP50_Pedestrian": 0.8451}, "RT-DETR-L": {"mAP50": 0.7314, "mAP50-95": 0.5068, "AP50_Car": 0.9291, "AP50_Bus": 0.882, "AP50_Truck": 0.4809, "AP50_Motorcycle": 0.8382, "AP50_Bicycle": 0.1557, "AP50_Rickshaw": 0.9007, "AP50_Mini-Truck": 0.6452, "AP50_CNG": 0.8952, "AP50_Pedestrian": 0.8554}, "Faster R-CNN": {"mAP50": 0.7112, "mAP50-95": 0.4648, "AP50_Car": 0.9255, "AP50_Bus": 0.873, "AP50_Truck": 0.425, "AP50_Motorcycle": 0.8028, "AP50_Bicycle": 0.1386, "AP50_Rickshaw": 0.8807, "AP50_Mini-Truck": 0.6189, "AP50_CNG": 0.8972, "AP50_Pedestrian": 0.8391}, "EfficientDet-D3": {"mAP50": 0.6654, "mAP50-95": 0.4358, "AP50_Car": 0.8971, "AP50_Bus": 0.8687, "AP50_Truck": 0.3652, "AP50_Motorcycle": 0.7561, "AP50_Bicycle": 0.1066, "AP50_Rickshaw": 0.8574, "AP50_Mini-Truck": 0.5989, "AP50_CNG": 0.8442, "AP50_Pedestrian": 0.6943}}

SKIP = {"tfp_dataset", "Bangladeshi Traffic Flow Dataset", "images", "labels", "tfp_8010", "runs"}
F = {"ORIG": None, "SH": None, "ALIGN": None, "SPLIT": None}
OLD_DETS, NEW_W = {}, {}
ALL5 = ["YOLOv8n", "YOLO11n", "RT-DETR-L", "Faster R-CNN", "EfficientDet-D3"]
NEW_NAMES = {"YOLOv8n_8010_best.pt": "YOLOv8n", "YOLO11n_8010_best.pt": "YOLO11n", "RT_DETR_L_8010_best.pt": "RT-DETR-L",
             "faster_rcnn_8020_FINAL.pth": "Faster R-CNN", "effdet_d3_8020_FINAL.pth": "EfficientDet-D3"}
CACHE = None
OLD_NAMES = {"dets_YOLOv8n.pkl": "YOLOv8n", "dets_YOLO11n.pkl": "YOLO11n", "dets_RT_DETR_L.pkl": "RT-DETR-L",
             "dets_Faster_R_CNN.pkl": "Faster R-CNN", "dets_EfficientDet_D3.pkl": "EfficientDet-D3"}
for root, dirs, files in os.walk("/kaggle/input"):
    if F["ORIG"] is None and "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            F["ORIG"] = c
    if F["SH"] is None and "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            F["SH"] = c
    dirs[:] = [d for d in dirs if d not in SKIP]
    for f in files:
        p = os.path.join(root, f)
        if f == "alignment_cache.pkl":
            F["ALIGN"] = p
        elif f == "split_8010_lists.json":
            F["SPLIT"] = p
        elif f in NEW_NAMES:
            NEW_W[NEW_NAMES[f]] = p
        elif f in OLD_NAMES:
            OLD_DETS[OLD_NAMES[f]] = p
        elif f == "dets_8010.pkl":
            CACHE = p
ORIG, SH = F["ORIG"], F["SH"]
print("\n" + "=" * 70)
for k, v in F.items():
    print(f"{'OK ' if v else 'MISSING':8s} {k:6s} {v or ''}")
CACHED = {}
if CACHE:
    CACHED = pickle.load(open(CACHE, "rb"))
    print("Ager 80/20 result (dets_8010.pkl):", CACHE, "->", sorted(CACHED["NEW_MAP"]))
for m in ALL5:
    src = "ager result theke" if m in CACHED.get("NEW_MAP", {}) else ("notun weight: OK" if m in NEW_W else "-- nai (skip hobe)")
    print(f"{m:16s} 80/20: {src:22s} | purono 70/20/10 dets: {'OK' if m in OLD_DETS else '-- nai'}")
print("=" * 70)
assert all(F.values()), "Upore MISSING input gulo jog koro"
MODELS = [m for m in ALL5 if (m in CACHED.get("NEW_MAP", {}) or m in NEW_W) and m in OLD_DETS]
assert MODELS, "kono 80/20 model pawa jayni - Faster R-CNN / EfficientDet training notebook er output Input e jog koro"
print("Tulona hobe:", MODELS)

In [ ]:
# CELL 3 (Part 1 er cache theke): camera-wise footpath zone + homography alignment (ekbar hisab kore cache)
ZONES = {
    (1, "DL", 1): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .56), (.8, .61), (1, .74), (1, .92), (.85, .79), (.75, .72), (.66, .63)]],
    (1, "DL", 2): [[(0, .45), (.27, .45), (.25, .55), (.195, .74), (.105, 1), (0, 1)],
                   [(.68, .57), (.78, .61), (.95, .69), (.95, .92), (.85, .84), (.7, .75), (.66, .68)]],
    (1, "DL", 3): [[(0, 0), (.25, 0), (.24, .12), (.2, .24), (.12, .36), (.03, .45), (0, .47)],
                   [(.72, .1), (.85, .1), (1, .3), (1, .65), (.9, .5), (.78, .32), (.72, .2)]],
    (1, "DL", 4): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .53), (.8, .57), (1, .68), (1, .88), (.85, .75), (.75, .68), (.66, .6)]],
    (1, "DL", 5): [[(0, .15), (.12, .15), (.12, .3), (.08, .45), (.03, .6), (0, .65)],
                   [(.6, .28), (1, .28), (1, .42), (.62, .38)]],
    (1, "SL", 1): [[(0, .5), (.2, .44), (.33, .42), (.32, .52), (.29, .62), (.16, 1), (0, 1)]],
    (1, "SL", 2): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 3): [[(0, .45), (.2, .38), (.35, .36), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 4): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 5): [[(0, .3), (.3, .3), (.32, .42), (.27, .6), (.2, .8), (.15, 1), (0, 1)]],
    (3, "DL", 1): [[(.55, .05), (.68, 0), (1, 0), (1, .5), (.76, .5), (.68, .4), (.6, .2)],
                   [(0, .05), (.2, .05), (.2, .2), (.12, .35), (0, .42)]],
    (3, "DL", 2): [[(.64, 0), (1, 0), (1, .55), (.9, .42), (.78, .27), (.7, .15)],
                   [(0, 0), (.2, 0), (.15, .2), (.1, .4), (.06, .6), (0, .7)]],
    (3, "DL", 3): [[(.6, 0), (1, 0), (1, .7), (.92, .55), (.8, .35), (.7, .2)],
                   [(0, 0), (.2, 0), (.13, .2), (.07, .4), (.03, .55), (0, .6)]],
    (3, "DL", 4): [[(.6, 0), (1, 0), (1, .45), (.85, .35), (.72, .2), (.62, .08)],
                   [(0, .05), (.22, .05), (.18, .25), (.12, .45), (.08, .6), (0, .65)]],
    (3, "DL", 5): [[(.68, 0), (1, 0), (1, 1), (.92, 1), (.8, .6), (.72, .3)],
                   [(0, .1), (.2, .1), (.15, .3), (.1, .5), (.05, .65), (0, .7)]],
    (3, "SL", 1): [[(.52, .2), (.62, .15), (1, .45), (1, 1), (.78, 1), (.68, .75), (.6, .55), (.54, .35)]],
    (3, "SL", 2): [[(.58, 0), (1, 0), (1, .72), (.9, .6), (.75, .35), (.62, .1)]],
    (3, "SL", 3): [[(.62, 0), (1, 0), (1, 1), (.92, 1), (.85, .6), (.75, .4), (.66, .2)]],
    (3, "SL", 4): [[(.62, 0), (1, 0), (1, .55), (.9, .4), (.75, .25), (.64, .12)]],
    (3, "SL", 5): [[(.58, 0), (1, 0), (1, .6), (.92, .62), (.8, .4), (.66, .2)]],
    (4, "DL", 1): [[(.55, .14), (.64, .24), (.8, .4), (.9, .49), (1, .56), (1, .7), (.9, .62), (.8, .54), (.64, .38), (.56, .22)],
                   [(0, .15), (.22, .15), (.2, .3), (.12, .45), (.05, .55), (0, .6)]],
    (4, "DL", 2): [[(.6, .4), (.7, .44), (.85, .5), (1, .56), (1, .72), (.85, .63), (.7, .55), (.6, .48)],
                   [(0, .4), (.2, .4), (.22, .5), (.18, .62), (.12, .75), (0, .85)]],
    (4, "DL", 3): [[(.55, .18), (.7, .32), (.85, .46), (1, .56), (1, .78), (.85, .64), (.75, .56), (.65, .5), (.58, .38), (.53, .25)],
                   [(0, .2), (.2, .2), (.25, .3), (.2, .45), (.12, .6), (.05, .75), (0, .8)]],
    (4, "DL", 4): [[(.58, .18), (.75, .25), (.9, .33), (1, .4), (1, .62), (.88, .52), (.75, .45), (.62, .35), (.56, .25)],
                   [(0, .15), (.22, .15), (.25, .25), (.2, .4), (.12, .5), (0, .55)]],
    (4, "DL", 5): [[(.58, .22), (.75, .32), (.9, .42), (1, .47), (1, .65), (.88, .58), (.75, .5), (.6, .35)],
                   [(0, .25), (.25, .25), (.28, .35), (.22, .55), (.15, .75), (.1, 1), (0, 1)]],
    (4, "SL", 1): [[(.48, .33), (.6, .42), (.75, .58), (.9, .72), (1, .78), (1, .92), (.88, .82), (.72, .68), (.58, .52), (.46, .4)]],
    (4, "SL", 2): [[(.48, .4), (.62, .5), (.75, .6), (.9, .72), (1, .78), (1, .95), (.88, .86), (.72, .74), (.6, .62), (.47, .48)]],
    (4, "SL", 3): [[(.5, .22), (.62, .28), (.72, .36), (.86, .52), (.95, .64), (1, .7), (1, .82), (.92, .76), (.84, .66), (.72, .54), (.6, .4), (.49, .3)]],
    (4, "SL", 4): [[(.53, .33), (.7, .45), (.85, .56), (1, .66), (1, 1), (.85, .9), (.75, .8), (.6, .6), (.52, .42)]],
    (4, "SL", 5): [[(0, .4), (.25, .4), (.35, .45), (.33, .6), (.3, .8), (.28, 1), (0, 1)]],
}

def key_of(p):
    m = re.search(r"Location (\d)", p)
    t = re.match(r"(\d)-LOC", os.path.basename(os.path.dirname(p)))
    lane = "DL" if "Double" in p else ("SL" if "Single" in p else None)
    return (int(m.group(1)), lane, int(t.group(1))) if (m and t and lane) else None

def stem_of(p):
    return os.path.splitext(os.path.basename(p))[0]

key_of_stem, folder_frames = {}, {}
for p in glob.glob(f"{ORIG}/Annotated Images/**/*", recursive=True):
    if p.lower().endswith(EXTS):
        k = key_of(p)
        key_of_stem[stem_of(p)] = k
        if k in ZONES:
            folder_frames.setdefault(k, []).append(p)
print(f"Camera folder: {len(folder_frames)} / 30")

W0, H0 = 640, 480
CORN = np.float32([[0, 0], [W0, 0], [W0, H0], [0, H0]]).reshape(-1, 1, 2)
MIN_INL, MIN_SCORE = 30, 0.15

def load(p):
    img = cv2.imread(p)
    if img is None:
        return None
    g = cv2.cvtColor(cv2.resize(img, (W0, H0)), cv2.COLOR_BGR2GRAY)
    gs = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.resize(g, (480, 360)))
    kp, d = cv2.SIFT_create(2000).detectAndCompute(gs, None)
    if d is None or len(kp) < 20:
        return None
    return {"g": g, "pts": np.float32([k.pt for k in kp]) * (W0 / 480), "d": d}

def sane(H):
    q = cv2.perspectiveTransform(CORN, H)
    area = cv2.contourArea(q) / (W0 * H0)
    return 0.3 < area < 3.0 and cv2.isContourConvex(q.astype(np.float32))

def homog(fa, fb):
    if fa is None or fb is None:
        return None, 0
    kn = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=32)).knnMatch(fa["d"], fb["d"], k=2)
    good = [x[0] for x in kn if len(x) == 2 and x[0].distance < 0.75 * x[1].distance]
    if len(good) < MIN_INL:
        return None, len(good)
    A = fa["pts"][[x.queryIdx for x in good]]
    B = fb["pts"][[x.trainIdx for x in good]]
    H, inl = cv2.findHomography(A, B, cv2.USAC_MAGSAC, 3.0, maxIters=5000, confidence=0.999)
    if H is None or inl is None:
        return None, 0
    n = int(inl.sum())
    if n < MIN_INL or not sane(H):
        return None, n
    return H / H[2, 2], n

def _grad(g):
    g = cv2.GaussianBlur(g.astype(np.float32), (0, 0), 1.5)
    return cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)

def score(fref, f, H):
    wa = cv2.warpPerspective(fref["g"], H, (W0, H0))
    wm = cv2.warpPerspective(np.full((H0, W0), 255, np.uint8), H, (W0, H0), flags=cv2.INTER_NEAREST)
    valid = cv2.erode(wm, np.ones((9, 9), np.uint8)) > 0
    if valid.mean() < 0.15:
        return 0.0
    ax, ay = _grad(wa)
    bx, by = _grad(f["g"])
    num = (ax * bx + ay * by)[valid].sum()
    den = np.sqrt((ax ** 2 + ay ** 2)[valid].sum() * (bx ** 2 + by ** 2)[valid].sum()) + 1e-6
    return float(num / den)

def zshift(k, H):
    d = []
    for poly in ZONES[k]:
        pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
        q = cv2.perspectiveTransform(pts, H).reshape(-1, 2) - pts.reshape(-1, 2)
        d += list(np.hypot(q[:, 0] / W0, q[:, 1] / H0))
    return float(np.median(d) * 100)

def fnum(p):
    m = re.search(r"frame(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else 0

def align_folder(k):
    fl = sorted(folder_frames[k])
    ref = fl[0]
    fr = load(ref)
    out = {stem_of(ref): (k, np.eye(3), "ref", 0.0, 1.0)}
    last = (fr, np.eye(3))
    for p in sorted(fl[1:], key=fnum):
        f = load(p)
        best = (None, "fail", 0.0)
        H, _ = homog(fr, f)
        if H is not None:
            s = score(fr, f, H)
            best = (H, "direct", s) if s >= MIN_SCORE else (None, "fail", s)
        if best[0] is None and last[0] is not None and last[0] is not fr:
            Hc, _ = homog(last[0], f)
            if Hc is not None:
                H2 = Hc @ last[1]
                H2 = H2 / H2[2, 2]
                if sane(H2):
                    s2 = score(fr, f, H2)
                    if s2 >= MIN_SCORE:
                        best = (H2, "chain", s2)
        H, how, s = best
        if H is None:
            out[stem_of(p)] = (k, None, "fail", np.nan, s)
        else:
            out[stem_of(p)] = (k, H, how, zshift(k, H), s)
            last = (f, H)
    return k, out


if F["ALIGN"]:
    AL = pickle.load(open(F["ALIGN"], "rb"))
    print(f"Alignment cache load holo: {F['ALIGN']} ({len(AL)} frame)")
else:
    t0 = time.time()
    AL = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        for i, (k, out) in enumerate(ex.map(align_folder, sorted(folder_frames))):
            AL.update(out)
            if (i + 1) % 5 == 0:
                print(f"  alignment: {i + 1}/{len(folder_frames)} folder ({time.time() - t0:.0f}s)")
    pickle.dump(AL, open(f"{WK}/alignment_cache.pkl", "wb"))
    print(f"Alignment done in {time.time() - t0:.0f}s, saved alignment_cache.pkl")

al = pd.DataFrame([{"stem": s, "key": str(v[0]), "how": v[2], "shift": v[3], "score": v[4]} for s, v in AL.items()])
al.to_csv(f"{OUTD}/frame_alignment.csv", index=False)
al["fail"] = al["how"] == "fail"
print(f"Mot {len(al)} frame | align hoyeche {100 - al['fail'].mean() * 100:.1f}%")
bad = al.groupby("key")["fail"].mean()
print("Jei camera folder e alignment fail > 5%:")
print((bad[bad > 0.05] * 100).round(1).to_string())

ZCACHE = {}
def zone_polys(stem):
    if stem in ZCACHE:
        return ZCACHE[stem]
    v = AL.get(stem)
    res = None
    if v is not None and v[1] is not None:
        res = []
        for poly in ZONES[v[0]]:
            pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
            res.append(Path(cv2.perspectiveTransform(pts, v[1]).reshape(-1, 2) / [W0, H0]))
    ZCACHE[stem] = res
    return res

In [ ]:
# CELL 4: test / 80-10-10 validation list, ground truth, helper
SPL = json.load(open(F["SPLIT"]))
path_of, label_of = {}, {}
for s in ("train", "val", "test"):
    for f in os.listdir(f"{SH}/{s}/images"):
        if f.lower().endswith(EXTS):
            st = os.path.splitext(f)[0]
            path_of[st] = f"{SH}/{s}/images/{f}"
            label_of[st] = f"{SH}/{s}/labels/{st}.txt"
TEST, VAL8 = sorted(SPL["test"]), sorted(SPL["val"])
assert set(TEST) == {os.path.splitext(f)[0] for f in os.listdir(f"{SH}/test/images")}, "test set mile na!"
print(f"test {len(TEST)} (= Sharmin test) | 80/10/10 validation {len(VAL8)} | train {len(SPL['train'])}")

def boxes_from_label(lp):
    out = []
    with open(lp) as fh:
        for line in fh:
            q = line.split()
            if len(q) >= 5:
                c, xc, yc, w, h = int(q[0]), *map(float, q[1:5])
                out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out
GT = {st: boxes_from_label(label_of[st]) for st in TEST + VAL8}

def foot(b):
    x0, y0, x1, y1 = b
    return (min(max((x0 + x1) / 2, 0.001), 0.999), min(max(y1, 0.001), 0.995))
def any_in_zone(boxes, stem):
    z = zone_polys(stem)
    if z is None:
        return False
    return any(any(P.contains_point(foot(b)) for P in z) for c, b in boxes if c not in PED)
def enc_pred(dets, stem, thr):
    return any_in_zone([(c, b) for c, b, s in dets if s >= thr], stem)
def enc_frames(stems):
    return [s for s in stems if key_of_stem.get(s) in ZONES and zone_polys(s) is not None]
ENC_T, ENC_V = enc_frames(TEST), enc_frames(VAL8)
GT_ET = np.array([any_in_zone(GT[s], s) for s in ENC_T])
GT_EV = [any_in_zone(GT[s], s) for s in ENC_V]
print(f"Encroachment frame: test {len(ENC_T)} (Part 2 te chilo 1,605) | validation {len(ENC_V)}")

def n_vehicle(boxes):
    return sum(1 for b in boxes if b[0] not in PED)
def lvl(c, Q=(7.0, 11.0)):
    return 0 if c <= Q[0] else (1 if c <= Q[1] else 2)
tr_counts = [n_vehicle(boxes_from_label(label_of[s])) for s in SPL["train"]]
Q8 = [float(x) for x in np.quantile(tr_counts, [0.33, 0.66])]
print(f"Congestion threshold - 80/10/10 train theke: Low <= {Q8[0]:g}, Medium <= {Q8[1]:g} (tulonar jonno same 7 / 11 use hobe)")
GT_LT = np.array([lvl(n_vehicle(GT[s])) for s in TEST])
GT_CT = np.array([n_vehicle(GT[s]) for s in TEST])
GT_LV = np.array([lvl(n_vehicle(GT[s])) for s in VAL8])

def summarize(p, g):
    p, g = np.asarray(p, dtype=bool), np.asarray(g, dtype=bool)
    n = len(p)
    tp, fp = int((p & g).sum()), int((p & ~g).sum())
    fn, tn = int((~p & g).sum()), int((~p & ~g).sum())
    po, pm, pg = (tp + tn) / n, (tp + fp) / n, (tp + fn) / n
    pc = pm * pg + (1 - pm) * (1 - pg)
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"agree_%": po * 100, "kappa": (po - pc) / (1 - pc) if pc < 1 else float("nan"),
            "f1_%": 2 * pr * rc / max(pr + rc, 1e-9) * 100, "recall_%": rc * 100, "precision_%": pr * 100}
print("ready")

In [ ]:
# CELL 5: notun model inference (test: mAP er jonno conf 0.001; validation: threshold bachai)
from ultralytics import YOLO, RTDETR
from torchmetrics.detection import MeanAveragePrecision

from PIL import Image
import torchvision.transforms.functional as TF
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def frcnn_runner(path):
    # same as Part 1 make_frcnn
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    m = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 10)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"] if "model_state_dict" in ck else ck)
    m.roi_heads.detections_per_img = 300
    m.to(DEV).eval()
    @torch.no_grad()
    def run_f(stems, conf, bs=8):
        m.roi_heads.score_thresh = min(conf, 0.05)
        out = {}
        for i in range(0, len(stems), bs):
            batch = stems[i:i + bs]
            ims = [Image.open(path_of[s]).convert("RGB") for s in batch]
            outs = m([TF.to_tensor(im).to(DEV) for im in ims])
            for s, im, o in zip(batch, ims, outs):
                W, H = im.size
                res = []
                for lab, (x1, y1, x2, y2), sc in zip(o["labels"].tolist(), o["boxes"].tolist(), o["scores"].tolist()):
                    if sc >= conf and lab >= 1:
                        res.append((lab - 1, [x1 / W, y1 / H, x2 / W, y2 / H], sc))
                out[s] = (W, H, res)
        return out
    return run_f

def effdet_runner(path):
    # same as Part 1 make_effdet (640 x 640, ImageNet mean/std, class id 1..9 -> 0..8), batched
    import effdet
    from effdet import get_efficientdet_config, EfficientDet
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model_state_dict" in sd:
        sd = sd["model_state_dict"]
    cfg = get_efficientdet_config("tf_efficientdet_d3")
    cfg.num_classes = 9
    cfg.image_size = (640, 640)
    try:
        cfg.max_det_per_image = 300
    except Exception:
        pass
    net = EfficientDet(cfg, pretrained_backbone=False)
    net.load_state_dict(sd, strict=True)
    Bench = getattr(effdet, "DetBenchPredict", None) or getattr(effdet, "DetBenchEval")
    try:
        bench = Bench(net, cfg)
    except TypeError:
        bench = Bench(net)
    bench.to(DEV).eval()
    @torch.no_grad()
    def run_e(stems, conf, bs=8):
        out = {}
        for i in range(0, len(stems), bs):
            batch = stems[i:i + bs]
            arrs, sizes = [], []
            for s in batch:
                img = Image.open(path_of[s]).convert("RGB")
                sizes.append(img.size)
                arrs.append((np.asarray(img.resize((640, 640), Image.BILINEAR), dtype=np.float32) / 255.0 - MEAN) / STD)
            o = bench(torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).contiguous().to(DEV))
            if isinstance(o, (list, tuple)):
                o = o[0]
            if o.dim() == 2:
                o = o.unsqueeze(0)
            for s, (W, H), det in zip(batch, sizes, o):
                res = []
                for x1, y1, x2, y2, sc, cl in det.tolist():
                    c = int(cl) - 1
                    if sc >= conf and 0 <= c <= 8:
                        res.append((c, [x1 / 640, y1 / 640, x2 / 640, y2 / 640], sc))
                out[s] = (W, H, res)
        return out
    return run_e

def load(name):
    m = RTDETR(NEW_W[name]) if name == "RT-DETR-L" else YOLO(NEW_W[name])
    cmap = {i: CLASSES.index(ALIAS[n.strip().lower()]) for i, n in m.names.items()}
    return m, cmap

def run(m, cmap, stems, conf, bs=16):
    out = {}
    for i in range(0, len(stems), bs):
        batch = stems[i:i + bs]
        res = m.predict(source=[path_of[s] for s in batch], conf=conf, imgsz=640, max_det=300, verbose=False)
        for s, r in zip(batch, res):
            H, W = r.orig_shape
            if r.boxes is None or len(r.boxes) == 0:
                out[s] = (W, H, [])
                continue
            out[s] = (W, H, list(zip([cmap[int(x)] for x in r.boxes.cls.tolist()], r.boxes.xyxyn.tolist(), r.boxes.conf.tolist())))
    return out

def to_px(boxes, W, H):
    return torch.tensor([[b[0] * W, b[1] * H, b[2] * W, b[3] * H] for b in boxes], dtype=torch.float32).reshape(-1, 4)

def map_on(dets, stems):
    m_all = MeanAveragePrecision(iou_type="bbox")
    m50 = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5], class_metrics=True)
    for s in stems:
        W, H, d = dets[s]
        pr = [{"boxes": to_px([b for c, b, sc in d], W, H), "scores": torch.tensor([sc for c, b, sc in d], dtype=torch.float32),
               "labels": torch.tensor([c for c, b, sc in d], dtype=torch.int64)}]
        tg = [{"boxes": to_px([b for c, b in GT[s]], W, H), "labels": torch.tensor([c for c, b in GT[s]], dtype=torch.int64)}]
        m_all.update(pr, tg)
        m50.update(pr, tg)
    ra, r5 = m_all.compute(), m50.compute()
    row = {"mAP50": float(ra["map_50"]), "mAP50-95": float(ra["map"])}
    for c, v in zip(r5["classes"].reshape(-1).tolist(), r5["map_per_class"].reshape(-1).tolist()):
        row[f"AP50_{CLASSES[int(c)]}"] = v
    return row

NEW_TEST, NEW_VAL, NEW_MAP = {}, {}, {}
for name in MODELS:
    t0 = time.time()
    if name in CACHED.get("NEW_MAP", {}):
        NEW_TEST[name], NEW_VAL[name], NEW_MAP[name] = CACHED["NEW_TEST"][name], CACHED["NEW_VAL"][name], CACHED["NEW_MAP"][name]
        print(f"[{name} 80/20] ager run er result nawa holo (abar chalano hoyni)")
        continue
    print(f"[{name} 80/20] {NEW_W[name]}")
    if name == "Faster R-CNN" or name == "EfficientDet-D3":
        runner = frcnn_runner(NEW_W[name]) if name == "Faster R-CNN" else effdet_runner(NEW_W[name])
        chk = runner(TEST[:4], 0.25)
        print("   check (4 chobi, conf 0.25): box count", [len(d) for (W, H, d) in chk.values()], "| GT", [len(GT[s]) for s in TEST[:4]])
        raw = runner(TEST, 0.001)
        NEW_MAP[name] = map_on(raw, TEST)
        NEW_TEST[name] = {s: [x for x in d if x[2] >= 0.10] for s, (W, H, d) in raw.items()}
        del raw
        NEW_VAL[name] = {s: d for s, (W, H, d) in runner(VAL8, 0.10).items()}
        del runner
    else:
        m, cmap = load(name)
        raw = run(m, cmap, TEST, 0.001)
        NEW_MAP[name] = map_on(raw, TEST)
        NEW_TEST[name] = {s: [x for x in d if x[2] >= 0.10] for s, (W, H, d) in raw.items()}
        del raw
        NEW_VAL[name] = {s: d for s, (W, H, d) in run(m, cmap, VAL8, 0.10).items()}
        del m
    gc.collect(); torch.cuda.empty_cache()
    print(f"   mAP50 {NEW_MAP[name]['mAP50']:.4f} (purono {MAP_OLD[name]['mAP50']:.4f}) | mAP50-95 {NEW_MAP[name]['mAP50-95']:.4f} "
          f"(purono {MAP_OLD[name]['mAP50-95']:.4f}) | {time.time() - t0:.0f}s")
OLD_TEST = {name: pickle.load(open(OLD_DETS[name], "rb"))[0]["test"] for name in MODELS}
pickle.dump({"NEW_TEST": NEW_TEST, "NEW_VAL": NEW_VAL, "NEW_MAP": NEW_MAP}, open(f"{WK}/dets_8020_all.pkl", "wb"))

In [ ]:
# CELL 6: purono vs notun - encroachment + congestion (same frame, jora dhore) + McNemar + bootstrap CI
from sklearn.metrics import f1_score
from scipy.stats import binomtest
rng = np.random.default_rng(42)
NB = 1000
IE = rng.integers(0, len(ENC_T), (NB, len(ENC_T)))
IC = rng.integers(0, len(TEST), (NB, len(TEST)))

def kappa_of(p, g):
    return summarize(p, g)["kappa"]
def ci(a):
    return f"{np.nanpercentile(a, 2.5):+.3f} to {np.nanpercentile(a, 97.5):+.3f}"
def mcnemar(a_ok, b_ok):
    n01, n10 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n01, n10, (binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0)
def count_at(dets, s, t):
    return sum(1 for c, b, sc in dets.get(s, []) if sc >= t and c not in PED)

rows, cls_rows, sweeps = [], [], {}
for name in MODELS:
    # threshold: notun model 80/10/10 validation e; purono model Part 2 er threshold
    sw = pd.DataFrame([{"thr": t, **summarize([enc_pred(NEW_VAL[name].get(s, []), s, t) for s in ENC_V], GT_EV)} for t in THRS])
    sw.to_csv(f"{OUTD}/enc_val_sweep_{name.replace('-', '_')}_8010.csv", index=False)
    sweeps[name] = sw
    te_new = float(sw.loc[sw["kappa"].idxmax(), "thr"])
    cs = [(t, f1_score(GT_LV, [lvl(count_at(NEW_VAL[name], s, t)) for s in VAL8], average="macro")) for t in THRS]
    tc_new = max(cs, key=lambda x: x[1])[0]

    pe_old = np.array([enc_pred(OLD_TEST[name].get(s, []), s, ENC_OLD[name]) for s in ENC_T])
    pe_new = np.array([enc_pred(NEW_TEST[name].get(s, []), s, te_new) for s in ENC_T])
    cnt_old = np.array([count_at(OLD_TEST[name], s, CONG_OLD[name]) for s in TEST])
    cnt_new = np.array([count_at(NEW_TEST[name], s, tc_new) for s in TEST])
    pl_old, pl_new = np.array([lvl(c) for c in cnt_old]), np.array([lvl(c) for c in cnt_new])

    E_old, E_new = summarize(pe_old, GT_ET), summarize(pe_new, GT_ET)
    dk = [kappa_of(pe_new[ix], GT_ET[ix]) - kappa_of(pe_old[ix], GT_ET[ix]) for ix in IE]
    da = [((pl_new[ix] == GT_LT[ix]).mean() - (pl_old[ix] == GT_LT[ix]).mean()) * 100 for ix in IC]
    e01, e10, ep = mcnemar(pe_new == GT_ET, pe_old == GT_ET)
    c01, c10, cp = mcnemar(pl_new == GT_LT, pl_old == GT_LT)
    for tag, Mp, E, thr_e, thr_c, pl, cnt in [("70/20/10", MAP_OLD[name], E_old, ENC_OLD[name], CONG_OLD[name], pl_old, cnt_old),
                                             ("80/10/10", NEW_MAP[name], E_new, te_new, tc_new, pl_new, cnt_new)]:
        rows.append({"model": name, "split": tag, "mAP50": Mp["mAP50"], "mAP50-95": Mp["mAP50-95"],
                     "enc_thr": thr_e, "enc_kappa": E["kappa"], "enc_f1_%": E["f1_%"], "enc_recall_%": E["recall_%"],
                     "enc_precision_%": E["precision_%"], "cong_thr": thr_c, "cong_acc_%": (pl == GT_LT).mean() * 100,
                     "cong_macro_f1_%": f1_score(GT_LT, pl, average="macro") * 100, "count_MAE": np.abs(cnt - GT_CT).mean(),
                     "delta_kappa_CI": ci(dk) if tag == "80/10/10" else "",
                     "enc_mcnemar_new_better/old_better/p": f"{e01}/{e10}/{ep:.3f}" if tag == "80/10/10" else "",
                     "delta_cong_acc_CI": ci(da) if tag == "80/10/10" else "",
                     "cong_mcnemar_new_better/old_better/p": f"{c01}/{c10}/{cp:.3f}" if tag == "80/10/10" else ""})
    for c in CLASSES:
        cls_rows.append({"model": name, "class": c, "AP50_70/20/10": MAP_OLD[name].get(f"AP50_{c}"),
                         "AP50_80/10/10": NEW_MAP[name].get(f"AP50_{c}")})

R = pd.DataFrame(rows)
R.to_csv(f"{OUTD}/compare_70_vs_80.csv", index=False)
C = pd.DataFrame(cls_rows)
C["change"] = C["AP50_80/10/10"] - C["AP50_70/20/10"]
C.to_csv(f"{OUTD}/classwise_AP50_70_vs_80.csv", index=False)
print("Sanity: purono model er kappa Part 2 er sathe mile? ", {n: round(r["enc_kappa"], 4) for n, r in zip(R["model"], R.to_dict("records")) if r["split"] == "70/20/10"})
print("\n" + "=" * 90 + "\n70/20/10 vs 80/10/10 - SAME test set (2,368 images; encroachment 1,605 frames)\n" + "=" * 90)
print(R.round(4).T.to_string())
print("\nClass-wise AP50 (change = 80/10/10 - 70/20/10):")
print(C.pivot(index="class", columns="model", values="change").reindex(CLASSES).round(3).to_string())

In [ ]:
# CELL 7: zip
zp = f"{WK}/eval_8020_all_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTD)):
        z.write(os.path.join(OUTD, f), f)
print("Saved:", zp)